# Waseel — Full Integrated Pipeline (Single Colab Notebook)

**A voice-first medication companion for visually impaired Saudi Arabic speakers.**

This notebook merges all six of the team's Colab blocks (System Setup, OCR,
Arabic LLM + TTS, JSON Database, Alert Logic, and Demo Integration) into one
self-contained pipeline, plus the biometric sign-in / caregiver-sync feature
from the Privacy & Security scenario in the Technical Architecture deck.

## ITU alignment (see `Waseel Team Plan.pdf` and `Technical-Architecture-and-Product-Roadmap.pdf`)
- **ITU-T Y.3172** — the pipeline is kept in explicit SRC → PP → M → SINK stages
  (camera/voice input → preprocessing → OCR/LLM inference → voice playback / SMS alert),
  annotated in the section headers below.
- **ITU AI Readiness 2.0 — Human Interface & AI for Inclusion dimensions** — every
  patient-facing interaction is voice-first, in Saudi dialect Arabic, and the LLM
  never invents dosage/warning content — it only paraphrases fields already present
  in `knowledge_base.json` (grounding, to avoid hallucination risk in medical guidance).
- **SFDA compliance rules** (defined in Code Block 1 below) are actively used, not just
  printed for show: every AI-generated explanation is logged to an **audit trail**
  (SFDA-REG-003) and carries a **confidence score** (SFDA-REG-002).
- **Privacy** — raw captured photos and audio used for biometric verification are
  deleted immediately after processing; nothing but derived results is persisted.

## Important honesty note on biometrics
A Colab notebook running in a browser **cannot** access a phone's real Face ID /
Touch ID hardware (Secure Enclave) — that only exists inside a native iOS/Android
app or a WebAuthn platform authenticator. For this demo:
- **Face verification is real** (webcam capture + OpenCV face matching) — good enough
  to demo the flow, not production-grade biometric security.
- **Fingerprint is a documented stub** that simulates success after a short delay.
  In the real Streamlit/mobile app, replace both with native Touch ID/Face ID or
  WebAuthn calls.

## Section 0 — Setup: install dependencies

In [ ]:
# !pip install -q easyocr gTTS anthropic opencv-contrib-python-headless SpeechRecognition

In [ ]:
import os
import io
import re
import json
import uuid
import time
import random
import logging
from datetime import datetime, timezone, timedelta
from difflib import SequenceMatcher
from typing import Optional, List, Dict, Any

import cv2
import numpy as np

## Section 1 — Code Block 1: Compliance Rules + Biometric Sign-In
**ITU-T Y.3172 stage: SRC (identity/consent gate before any patient data flows).**

This keeps the original Code Block 1 (SFDA compliance rule definitions), fixes the
`datetime.utcnow()` deprecation warning, and adds the biometric sign-in / caregiver
sync feature described in the Privacy & Security scenario:
`Sign-in (biometric) -> Sync request -> re-verify biometric -> 6-digit code (10 min
expiry) -> caregiver redeems code`.

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - [WASEEL-ITU-Y.3172] - %(levelname)s - %(message)s'
)
logger = logging.getLogger("WaseelCore")


class ITUDataCollector:
    """ITU-T Y.3172 Component: Ingests raw SFDA dataset rules and policy inputs."""
    def __init__(self, dataset_name: str):
        self.dataset_name = dataset_name
        logger.info(f"Initialized ITU Data Collector for: {self.dataset_name}")

    def collect_sfda_rules(self) -> Dict[str, Any]:
        sfda_payload = {
            "metadata": {
                "source": "SFDA Regulatory Knowledge Base",
                "version": "2026.1",
                "timestamp": datetime.now(timezone.utc).isoformat(),
            },
            "rules": [
                {
                    "rule_id": "SFDA-REG-001",
                    "category": "Data Privacy & Governance",
                    "description": "Patient and clinical data must be stored within sovereign boundaries.",
                    "compliance_weight": 0.35,
                },
                {
                    "rule_id": "SFDA-REG-002",
                    "category": "Algorithmic Transparency",
                    "description": "AI diagnostic models must provide explainable confidence scores.",
                    "compliance_weight": 0.40,
                },
                {
                    "rule_id": "SFDA-REG-003",
                    "category": "Safety & Auditability",
                    "description": "All automated recommendations require immutable audit logging.",
                    "compliance_weight": 0.25,
                },
            ],
        }
        logger.info(f"Successfully collected {len(sfda_payload['rules'])} SFDA regulatory rules.")
        return sfda_payload


class ITUPreprocessing:
    """ITU-T Y.3172 Component: Validates and formats raw data for ML Policy evaluation."""
    def process(self, raw_data: Dict[str, Any]) -> List[Dict[str, Any]]:
        logger.info("Cleaning and normalizing SFDA compliance vectors...")
        rules = raw_data.get("rules", [])
        for rule in rules:
            rule["status"] = "READY_FOR_EVALUATION"
        return rules


# --- Biometric sign-in -------------------------------------------------------

FACE_CASCADE = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")


def detect_face(gray_image, size=(200, 200)):
    """Returns a normalized, cropped face region, or None if no face is found."""
    faces = FACE_CASCADE.detectMultiScale(gray_image, scaleFactor=1.1, minNeighbors=5, minSize=(80, 80))
    if len(faces) == 0:
        return None
    x, y, w, h = max(faces, key=lambda f: f[2] * f[3])  # largest face in frame
    return cv2.resize(gray_image[y:y + h, x:x + w], size)


class BiometricAuth:
    """
    Demo-grade biometric layer for the Colab prototype.

    Face verification: real (webcam capture -> OpenCV LBPH face matching).
    Fingerprint: a documented stub — see module docstring above for why.

    Privacy: raw captured images are deleted immediately after a face
    template/verification result is extracted (only the trained model,
    not the photo, is kept), per ITU AI for Inclusion privacy-minimization
    guidance ("only the key points are transferred, not the users' images").
    """

    def __init__(self, model_dir: str = "biometric_models"):
        os.makedirs(model_dir, exist_ok=True)
        self.model_dir = model_dir

    def _model_path(self, patient_id: str) -> str:
        return os.path.join(self.model_dir, f"{patient_id}_face.yml")

    def enroll_face(self, patient_id: str, num_samples: int = 5) -> bool:
        samples = []
        for i in range(num_samples):
            print(f"[Enrollment] Photo {i + 1}/{num_samples} — look at the camera...")
            path = take_photo(filename=f"enroll_{patient_id}_{i}.jpg")
            image = cv2.imread(path)
            gray = to_grayscale(image)
            face = detect_face(gray)
            os.remove(path)  # do not retain raw biometric images
            if face is not None:
                samples.append(face)
        if len(samples) < 2:
            raise RuntimeError("Could not capture enough clear face samples — try better lighting.")
        recognizer = cv2.face.LBPHFaceRecognizer_create()
        recognizer.train(samples, np.zeros(len(samples), dtype=np.int32))
        recognizer.save(self._model_path(patient_id))
        print(f"Face enrolled for patient {patient_id}.")
        return True

    def verify_face(self, patient_id: str, distance_threshold: float = 60.0):
        model_path = self._model_path(patient_id)
        if not os.path.exists(model_path):
            raise RuntimeError(f"No enrolled face model for {patient_id}. Call enroll_face() first.")
        recognizer = cv2.face.LBPHFaceRecognizer_create()
        recognizer.read(model_path)

        path = take_photo(filename=f"verify_{patient_id}.jpg")
        image = cv2.imread(path)
        gray = to_grayscale(image)
        face = detect_face(gray)
        os.remove(path)

        if face is None:
            return False, None
        _, distance = recognizer.predict(face)
        return distance <= distance_threshold, distance

    def verify_fingerprint(self, patient_id: str) -> bool:
        """
        STUB: a browser/Colab session has no access to a physical fingerprint
        sensor. The real app calls the device's native Touch ID (Secure Enclave)
        or a WebAuthn platform authenticator here instead.
        """
        print("[DEMO STUB] Simulating Touch ID scan (native app -> Secure Enclave / WebAuthn)...")
        time.sleep(1.2)
        return True

    def sign_in(self, patient_id: str, method: str = "face") -> bool:
        """Top-level sign-in gate used at app launch."""
        if method == "fingerprint":
            ok = self.verify_fingerprint(patient_id)
        else:
            ok, distance = self.verify_face(patient_id)
            print(f"Face match distance: {distance} (lower = closer match, threshold 60)" if distance is not None
                  else "No face detected.")
        print("✅ Sign-in verified." if ok else "❌ Sign-in failed — identity not verified.")
        return ok

## Section 2 — Code Block 2: OCR + Capture Guidance
**ITU-T Y.3172 stage: SRC → PP (camera input, then image preprocessing).**

Adds a lightweight, heuristic capture-quality check (blur + brightness) that gives
the patient spoken Arabic guidance ("get closer", "lighting is weak") before OCR
runs — this is a rule-based heuristic, not a separate trained vision model, which
is an appropriate scope for a hackathon demo.

In [ ]:
def take_photo(filename: str = "capture.jpg", quality: float = 0.9) -> str:
    """Opens the device camera in Colab and saves a captured frame to disk."""
    from IPython.display import display, Javascript
    from google.colab.output import eval_js
    from base64 import b64decode

    js = Javascript('''
        async function takePhoto(quality) {
            const div = document.createElement('div');
            const capture = document.createElement('button');
            capture.textContent = 'Capture';
            div.appendChild(capture);

            const video = document.createElement('video');
            video.style.display = 'block';

            const constraints = {
                video: { width: {ideal: 1920}, height: {ideal: 1080}, facingMode: 'environment' }
            };
            const stream = await navigator.mediaDevices.getUserMedia(constraints);

            document.body.appendChild(div);
            div.appendChild(video);
            video.srcObject = stream;
            await video.play();

            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
            await new Promise((resolve) => capture.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getVideoTracks()[0].stop();
            div.remove();
            return canvas.toDataURL('image/jpeg', quality);
        }
    ''')
    try:
        display(js)
        data = eval_js('takePhoto({})'.format(quality))
        binary = b64decode(data.split(',')[1])
        with open(filename, 'wb') as f:
            f.write(binary)
        return filename
    except Exception as e:
        # Fallback for demo environments without camera access (e.g. desktop
        # Colab, judges' laptops without a webcam): let the user upload a photo
        # instead, so the demo still runs end to end.
        print(f"Camera capture unavailable ({e}). Please upload a photo instead.")
        from google.colab import files
        uploaded = files.upload()
        uploaded_name = list(uploaded.keys())[0]
        os.rename(uploaded_name, filename)
        return filename


def to_grayscale(image):
    if len(image.shape) == 3:
        return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return image


def upscale(image, factor=2):
    return cv2.resize(image, None, fx=factor, fy=factor, interpolation=cv2.INTER_CUBIC)


def enhance_contrast(image):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(image)


def preprocess_image(image):
    gray = to_grayscale(image)
    h, w = gray.shape
    if max(h, w) < 1000:
        gray = upscale(gray)
    return enhance_contrast(gray)


GUIDANCE_MESSAGES_AR = {
    "blurry": "الصورة مو واضحة، حاول تثبّت الجوال وقرّب شوي على ملصق الدواء.",
    "too_dark": "الإضاءة ضعيفة، حاول تصوّر في مكان أضوى.",
    "too_bright": "في وهج زايد على الصورة، بعّد شوي عن الضوء المباشر.",
}


def assess_capture_quality(image) -> list:
    gray = to_grayscale(image)
    blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()
    brightness = gray.mean()
    issues = []
    if blur_score < 50:
        issues.append("blurry")
    if brightness < 60:
        issues.append("too_dark")
    elif brightness > 200:
        issues.append("too_bright")
    return issues


def capture_with_guidance(max_attempts: int = 3) -> str:
    """Camera capture loop with spoken Arabic guidance — this is the 'AI guidance
    so the user knows where to capture' from the patient scenario."""
    for attempt in range(1, max_attempts + 1):
        photo_path = take_photo(filename=f"med_capture_{attempt}.jpg")
        image = cv2.imread(photo_path)
        issues = assess_capture_quality(image)
        if not issues:
            return photo_path
        message = " ".join(GUIDANCE_MESSAGES_AR[i] for i in issues)
        print(f"⚠️ Capture guidance: {message}")
        guidance_audio = synthesize_speech(message, output_path=f"guidance_{attempt}.mp3")
        play_audio(guidance_audio)
        os.remove(photo_path)
    print("Proceeding with the last capture despite quality warnings.")
    return take_photo(filename="med_capture_final.jpg")


_ocr_reader = None


def get_ocr_reader():
    global _ocr_reader
    if _ocr_reader is None:
        import easyocr
        _ocr_reader = easyocr.Reader(['ar', 'en'], gpu=False)
    return _ocr_reader


def extract_medicine_details(image) -> dict:
    reader = get_ocr_reader()
    results = reader.readtext(image)

    lines = []
    for bbox, text, conf in results:
        text = text.strip()
        if not text:
            continue
        top_left, top_right, bottom_right, bottom_left = bbox
        height = bottom_right[1] - top_right[1]
        lines.append({"text": text, "height": height, "confidence": conf})

    lines.sort(key=lambda x: x["height"], reverse=True)
    return {
        "name": lines[0]["text"] if lines else "",
        "name_confidence": lines[0]["confidence"] if lines else 0.0,
        "avg_confidence": sum(l["confidence"] for l in lines) / len(lines) if lines else 0.0,
        "all_detected": [l["text"] for l in lines],
    }

## Section 3 — Voice Input (record + Arabic transcription)
**ITU-T Y.3172 stage: SRC → PP (voice input, then audio-to-text preprocessing).**

Lets the patient log a dose or answer a check-in "by voice" instead of the camera,
per the scenario requirement.

In [ ]:
def record_audio(filename: str = "voice_input.wav", seconds: int = 5) -> str:
    """Records microphone audio in Colab and converts it to a 16kHz mono WAV."""
    from IPython.display import display, Javascript
    from google.colab.output import eval_js
    from base64 import b64decode

    js = Javascript('''
        async function recordAudio(seconds) {
            const stream = await navigator.mediaDevices.getUserMedia({audio: true});
            const recorder = new MediaRecorder(stream);
            const chunks = [];
            recorder.ondataavailable = e => chunks.push(e.data);
            recorder.start();
            await new Promise(resolve => setTimeout(resolve, seconds * 1000));
            recorder.stop();
            await new Promise(resolve => recorder.onstop = resolve);
            stream.getTracks().forEach(t => t.stop());
            const blob = new Blob(chunks, {type: 'audio/webm'});
            const buffer = await blob.arrayBuffer();
            const bytes = new Uint8Array(buffer);
            let binary = '';
            for (let i = 0; i < bytes.length; i++) binary += String.fromCharCode(bytes[i]);
            return btoa(binary);
        }
    ''')
    try:
        display(js)
        data = eval_js(f'recordAudio({seconds})')
        binary = b64decode(data)
        webm_path = filename.replace(".wav", ".webm")
        with open(webm_path, "wb") as f:
            f.write(binary)
        os.system(f'ffmpeg -y -i "{webm_path}" -ar 16000 -ac 1 "{filename}" -loglevel quiet')
        os.remove(webm_path)
        return filename
    except Exception as e:
        print(f"Microphone capture unavailable ({e}). Please type the message instead.")
        return None


def transcribe_arabic_speech(audio_path: Optional[str], language: str = "ar-SA") -> str:
    if audio_path is None:
        return input("اكتب رسالتك هنا (النص بدل الصوت): ")
    import speech_recognition as sr
    recognizer = sr.Recognizer()
    with sr.AudioFile(audio_path) as source:
        audio = recognizer.record(source)
    os.remove(audio_path)  # don't retain raw voice recordings after transcription
    try:
        return recognizer.recognize_google(audio, language=language)
    except sr.UnknownValueError:
        return ""
    except sr.RequestError as e:
        print(f"Speech recognition service error: {e}")
        return ""

## Section 4 — Code Block 4: Knowledge Base loading + matching
**ITU-T Y.3172 stage: PP → M (matching OCR/voice text to a grounded drug record).**

In [ ]:
def load_knowledge_base(kb_path: str = "waseel_knowledge_base.json") -> list:
    with open(kb_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    return raw_data["medications"]


def get_trade_name(entry: dict, lang: str = "en") -> str:
    return entry.get("tradeName", {}).get(lang, "")


def find_best_match_text(query_lines: list, knowledge_base: list, min_score: float = 0.5):
    """Used for both OCR lines and a transcribed voice query."""
    best_entry, best_score, best_line = None, 0.0, ""
    for line in query_lines:
        line_clean = line.strip().lower()
        if not line_clean:
            continue
        for entry in knowledge_base:
            for lang in ("en", "ar"):
                trade_name = get_trade_name(entry, lang).strip().lower()
                if not trade_name:
                    continue
                score = SequenceMatcher(None, line_clean, trade_name).ratio()
                if score > best_score:
                    best_score, best_entry, best_line = score, entry, line
    if best_score >= min_score:
        return best_entry, best_score, best_line
    return None, best_score, best_line


def assess_risk_level(entry: dict) -> str:
    """
    HEURISTIC: the knowledge base has no explicit risk_level field. Until the
    team defines one from real SFDA/SDI severity data, treat presence of a
    'warning' field, or an Rx legal status with seriousWarningSigns, as high risk.
    """
    if entry.get("warning"):
        return "high"
    if entry.get("legalStatus") == "Rx" and entry.get("seriousWarningSigns"):
        return "high"
    return "low"

## Section 5 — Code Block 3: LLM (Saudi dialect) + TTS
**ITU-T Y.3172 stage: M → SINK (grounded inference, then spoken output).**

Rebuilt cleanly from Abrar's design (the PDF copy of this code had its Arabic
text and f-strings corrupted by PDF text extraction). Grounding rule preserved:
the LLM only paraphrases fields already present in the matched knowledge-base
entry — it never invents dosage or warning information.

A DEMO_MODE fallback (template-based, still fully grounded) keeps the whole
app usable at the hackathon even without an API key or internet connection.

In [ ]:
SYSTEM_PROMPT = """أنت "مساعد وصيل"، مساعد صوتي طبي يتكلم باللهجة السعودية البسيطة.
مستخدمك شخص كفيف أو ضعيف البصر، لذا:
- استخدم جمل قصيرة وواضحة (لا تتجاوز الجملة 15 كلمة).
- تجنب المصطلحات الطبية المعقدة، واشرح أي مصطلح لازم بكلمات بسيطة.
- اذكر المعلومات بالترتيب: اسم الدواء ثم الجرعة ثم متى يؤخذ ثم تحذيرات مهمة.
- لا تخمن أي معلومة دوائية غير موجودة في البيانات المعطاة لك.
- إذا كانت الجرعة أو التحذير يصنف "عالي الخطورة"، انبه المستخدم بوضوح وأخبره أنه سيتم تنبيه مرافقه.
- تكلم بأسلوب ودود ومطمئن، مثل شخص يشرح لصديق كبير بالسن.
- لا تستخدم رموز أو اختصارات لاتينية في ردك لأن الرد سيتحول إلى صوت.
"""

FALLBACK_TEXT = (
    "ما قدرت أتأكد من اسم الدواء بوضوح من الصورة. "
    "ممكن تصور الغلاف مرة ثانية بإضاءة أفضل، أو تقول لي اسم الدواء بصوتك؟"
)


def build_user_prompt(entry: dict) -> str:
    trade_name = get_trade_name(entry, "ar") or get_trade_name(entry, "en")
    generic_name = entry.get("genericName", {}).get("ar") or entry.get("genericName", {}).get("en", "")
    dosage_form = entry.get("dosageForm", "")
    dosage_instructions = entry.get("dosageInstructions_saudi_dialect", "")
    warning = entry.get("warning", "لا يوجد")
    risk_level = assess_risk_level(entry)

    return f"""معلومات الدواء (من قاعدة بيانات الهيئة العامة للغذاء والدواء SFDA):
- الاسم التجاري: {trade_name}
- الاسم العلمي: {generic_name}
- الشكل الصيدلاني: {dosage_form}
- الجرعة الموصى بها: {dosage_instructions}
- تحذيرات: {warning}
- مستوى الخطورة: {risk_level}

اشرح هذا الدواء للمستخدم الكفيف بلهجة سعودية بسيطة وواضحة، باستخدام أسلوب صوتي مناسب للقراءة الآلية (TTS)."""


def call_llm(client, model_name: str, system_prompt: str, user_prompt: str, demo_fallback: str) -> str:
    """Wraps every LLM call with a DEMO_MODE fallback so the app 'just works'
    at the hackathon even with no ANTHROPIC_API_KEY / no internet."""
    if client is None:
        print("[DEMO MODE] No live LLM configured — using a grounded template response.")
        return demo_fallback
    try:
        response = client.messages.create(
            model=model_name, max_tokens=400,
            system=system_prompt, messages=[{"role": "user", "content": user_prompt}],
        )
        return response.content[0].text
    except Exception as e:
        print(f"[DEMO MODE] LLM call failed ({e}) — falling back to a grounded template response.")
        return demo_fallback


def template_explanation(entry: dict) -> str:
    """Grounded, template-based explanation used when no live LLM is available."""
    trade_name = get_trade_name(entry, "ar") or get_trade_name(entry, "en")
    dosage_instructions = entry.get("dosageInstructions_saudi_dialect", "")
    risk_note = ""
    if assess_risk_level(entry) == "high":
        risk_note = " انتبه، هذا الدواء يحتاج متابعة، وراح أرسل تنبيه لمرافقك."
    return f"الدواء اللي معك هو {trade_name}. {dosage_instructions}{risk_note}"


def get_medication_explanation(matched_entry: Optional[dict], client, model_name: str):
    if matched_entry is None:
        return FALLBACK_TEXT, None, 0.0
    user_prompt = build_user_prompt(matched_entry)
    fallback = template_explanation(matched_entry)
    explanation_text = call_llm(client, model_name, SYSTEM_PROMPT, user_prompt, fallback)
    # SFDA-REG-002 (explainable confidence score): a simple, transparent proxy —
    # 1.0 when grounded in a matched KB record, 0.0 for the ungrounded fallback.
    confidence_score = 1.0
    return explanation_text, matched_entry, confidence_score


def synthesize_speech(text: str, output_path: str = "waseel_output.mp3") -> Optional[str]:
    try:
        from gtts import gTTS
        gTTS(text=text, lang="ar", slow=True).save(output_path)
        return output_path
    except Exception as e:
        print(f"[DEMO MODE] TTS unavailable ({e}) — showing text only:\n{text}")
        return None


def play_audio(audio_path: Optional[str]):
    if not audio_path:
        return
    try:
        from IPython.display import Audio, display
        display(Audio(audio_path, autoplay=False))
    except ImportError:
        pass

## Section 6 — Code Block 4/5: Data Layer + Notifications (extended)
**ITU-T Y.3172 stage: SINK (persisted state + caregiver notification).**

This is `waseel_db.py`'s `WaseelDB` class, kept logically identical, **plus**
three additions needed for the new scenario (see the "what changed" note in the
chat reply): `pairing_codes` (biometric caregiver sync), `scheduled_reminders`
(medication alarms), and `audit_log` (SFDA-REG-003 auditability).

In [ ]:
DB_PATH_DEFAULT = "database_schema.json"


class WaseelDB:
    def __init__(self, db_path: str = DB_PATH_DEFAULT):
        self.db_path = db_path
        if not os.path.exists(self.db_path):
            self._init_empty_db()
        self.data = self._load()

    def _init_empty_db(self):
        empty = {
            "patients": [],
            "caregivers": [],
            "medication_logs": [],
            "alert_thresholds": [],
            "medication_knowledge_base": [],
            "pairing_codes": [],
            "scheduled_reminders": [],
            "audit_log": [],
        }
        with open(self.db_path, "w", encoding="utf-8") as f:
            json.dump(empty, f, ensure_ascii=False, indent=2)

    def _load(self) -> dict:
        with open(self.db_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        for table in ("pairing_codes", "scheduled_reminders", "audit_log"):
            data.setdefault(table, [])
        return data

    def _save(self):
        with open(self.db_path, "w", encoding="utf-8") as f:
            json.dump(self.data, f, ensure_ascii=False, indent=2)

    # ---------- Patient / Caregiver ----------

    def add_patient(self, name: str) -> str:
        patient_id = f"P-{uuid.uuid4().hex[:6].upper()}"
        self.data["patients"].append({"patient_id": patient_id, "name": name})
        self._save()
        return patient_id

    def get_patient(self, patient_id: str) -> Optional[dict]:
        return next((p for p in self.data["patients"] if p["patient_id"] == patient_id), None)

    def add_caregiver(self, name: str, contact_info: str, relationship: str,
                       linked_patient_id: str, severity_threshold: int = 6) -> str:
        caregiver_id = f"C-{uuid.uuid4().hex[:6].upper()}"
        self.data["caregivers"].append({
            "caregiver_id": caregiver_id,
            "name": name,
            "contact_info": contact_info,
            "relationship": relationship,
            "linked_patient_id": linked_patient_id,
            "sms_alert_settings": {
                "enabled": True,
                "missed_dose_alert": True,
                "symptom_severity_threshold": severity_threshold,
            },
        })
        self._save()
        return caregiver_id

    def get_caregivers_for_patient(self, patient_id: str) -> list:
        return [c for c in self.data["caregivers"] if c["linked_patient_id"] == patient_id]

    # ---------- Biometric caregiver sync (NEW) ----------

    def create_pairing_code(self, patient_id: str, expires_in_minutes: int = 10):
        code = f"{random.randint(0, 999999):06d}"
        now = datetime.now()
        expires_at = (now + timedelta(minutes=expires_in_minutes)).isoformat()
        # invalidate any previous unused codes for this patient
        self.data["pairing_codes"] = [c for c in self.data["pairing_codes"]
                                       if c["patient_id"] != patient_id or c["used"]]
        self.data["pairing_codes"].append({
            "code": code, "patient_id": patient_id,
            "created_at": now.isoformat(), "expires_at": expires_at, "used": False,
        })
        self._save()
        return code, expires_at

    def redeem_pairing_code(self, code: str, caregiver_name: str, contact_info: str, relationship: str) -> dict:
        entry = next((c for c in self.data["pairing_codes"] if c["code"] == code and not c["used"]), None)
        if not entry:
            return {"success": False, "reason": "invalid_or_used_code"}
        if datetime.now() > datetime.fromisoformat(entry["expires_at"]):
            return {"success": False, "reason": "expired"}
        entry["used"] = True
        self._save()
        caregiver_id = self.add_caregiver(caregiver_name, contact_info, relationship, entry["patient_id"])
        self.log_audit_event("caregiver_paired", entry["patient_id"],
                              {"caregiver_id": caregiver_id})
        return {"success": True, "caregiver_id": caregiver_id, "patient_id": entry["patient_id"]}

    # ---------- Medication Logs ----------

    def log_medication(self, patient_id: str, medication_id: str, medication_name: str,
                        scheduled_time: str, actual_time: Optional[str] = None,
                        status: str = "taken", symptoms_reported: str = "",
                        severity_score: int = 0, caregiver_visual_check_needed: bool = False) -> dict:
        delay_minutes = 0
        if actual_time and status == "taken":
            sched = datetime.fromisoformat(scheduled_time)
            actual = datetime.fromisoformat(actual_time)
            delay_minutes = max(0, int((actual - sched).total_seconds() // 60))

        log_entry = {
            "log_id": f"L-{uuid.uuid4().hex[:6].upper()}",
            "patient_id": patient_id,
            "medication_id": medication_id,
            "medication_name": medication_name,
            "scheduled_time": scheduled_time,
            "actual_time": actual_time,
            "status": status,
            "delay_minutes": delay_minutes,
            "symptoms_reported": symptoms_reported,
            "severity_score": severity_score,
            "caregiver_visual_check_needed": caregiver_visual_check_needed,
        }
        self.data["medication_logs"].append(log_entry)
        self._save()
        alert = self._check_alert(patient_id, log_entry)
        return {"log": log_entry, "alert_triggered": alert}

    def get_medication_history(self, patient_id: str) -> list:
        return [l for l in self.data["medication_logs"] if l["patient_id"] == patient_id]

    # ---------- Alert Thresholds ----------

    def set_alert_threshold(self, patient_id: str, missed_dose_minutes: int = 30,
                             severity_threshold: int = 6):
        existing = next((a for a in self.data["alert_thresholds"] if a["patient_id"] == patient_id), None)
        if existing:
            existing["missed_dose_minutes_before_alert"] = missed_dose_minutes
            existing["symptom_severity_alert_threshold"] = severity_threshold
        else:
            self.data["alert_thresholds"].append({
                "patient_id": patient_id,
                "missed_dose_minutes_before_alert": missed_dose_minutes,
                "symptom_severity_alert_threshold": severity_threshold,
            })
        self._save()

    def _check_alert(self, patient_id: str, log_entry: dict) -> bool:
        threshold = next((a for a in self.data["alert_thresholds"] if a["patient_id"] == patient_id), None)
        if not threshold:
            return False
        triggered = (
            log_entry["status"] == "missed"
            or log_entry["severity_score"] >= threshold["symptom_severity_alert_threshold"]
            or log_entry.get("caregiver_visual_check_needed", False)
        )
        if triggered:
            self._send_sms_alert(patient_id, log_entry)
        return triggered

    def _send_sms_alert(self, patient_id: str, log_entry: dict):
        """Integration point for a real SMS provider (e.g. Twilio) later."""
        caregivers = self.get_caregivers_for_patient(patient_id)
        for c in caregivers:
            if c["sms_alert_settings"]["enabled"]:
                extra = " — يحتاج تأكيد بصري من مرافقه." if log_entry.get("caregiver_visual_check_needed") else ""
                print(f"[SMS ALERT] → {c['name']} ({c['contact_info']}): "
                      f"دواء {log_entry['medication_name']} — الحالة: {log_entry['status']}, "
                      f"شدة الأعراض: {log_entry['severity_score']}{extra}")
        self.log_audit_event("caregiver_sms_alert", patient_id, log_entry,
                              compliance_rule_ids=["SFDA-REG-003"])

    # ---------- Medication Knowledge Base (local mirror, optional) ----------

    def add_drug(self, trade_name: str, generic_name: str,
                 dosage_instructions_dialect: str, common_symptoms: list) -> str:
        medication_id = f"M-{uuid.uuid4().hex[:6].upper()}"
        self.data["medication_knowledge_base"].append({
            "medication_id": medication_id,
            "trade_name": trade_name,
            "generic_name": generic_name,
            "dosage_instructions_dialect": dosage_instructions_dialect,
            "common_symptoms": common_symptoms,
        })
        self._save()
        return medication_id

    def get_drug(self, medication_id: str) -> Optional[dict]:
        return next((d for d in self.data["medication_knowledge_base"]
                     if d["medication_id"] == medication_id), None)

    # ---------- Scheduled reminders (NEW) ----------

    def add_reminders(self, reminders: list):
        self.data["scheduled_reminders"].extend(reminders)
        self._save()

    def get_due_reminders(self, current_time: Optional[datetime] = None) -> list:
        current_time = current_time or datetime.now()
        due = []
        for r in self.data["scheduled_reminders"]:
            if not r["fired"] and datetime.fromisoformat(r["due_time"]) <= current_time:
                r["fired"] = True
                due.append(r)
        if due:
            self._save()
        return due

    # ---------- Audit log (NEW — SFDA-REG-003) ----------

    def log_audit_event(self, event_type: str, patient_id: str, details: dict,
                         compliance_rule_ids: Optional[list] = None) -> dict:
        entry = {
            "audit_id": f"A-{uuid.uuid4().hex[:6].upper()}",
            "timestamp": datetime.now().isoformat(),
            "event_type": event_type,
            "patient_id": patient_id,
            "details": details,
            "compliance_rule_ids": compliance_rule_ids or [],
        }
        self.data["audit_log"].append(entry)
        self._save()
        return entry

## Section 7 — Code Block 5: Medication Scheduler & Reminders
**ITU-T Y.3172 stage: M → SINK (derived schedule -> future voice/notification action).**

Parses a simple dosing frequency out of the Saudi-dialect instructions text (e.g.
"كل ٨ ساعات" or "٣ إلى ٤ مرات باليوم") to auto-build the next few reminder times.
In this Colab demo, "firing" a reminder means printing + speaking it when you call
`get_due_reminders()` — a production deployment would instead trigger a push
notification/OS alarm from a backend cron job at each `due_time`.

In [ ]:
ARABIC_INDIC_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
HOURS_INTERVAL_PATTERN = re.compile(r"كل\s*(\d+)\s*ساع")
TIMES_PER_DAY_PATTERN = re.compile(r"(\d+)\s*(?:إلى\s*(\d+)\s*)?مرات?\s*(?:في\s*اليوم|باليوم)")


def parse_dosing_interval_hours(dosage_text: str, default_hours: int = 8) -> int:
    text = dosage_text.translate(ARABIC_INDIC_DIGITS)
    m = HOURS_INTERVAL_PATTERN.search(text)
    if m:
        return int(m.group(1))
    m = TIMES_PER_DAY_PATTERN.search(text)
    if m:
        low = int(m.group(1))
        high = int(m.group(2)) if m.group(2) else low
        times_per_day = max(1, round((low + high) / 2))
        return max(1, round(24 / times_per_day))
    return default_hours


def schedule_reminders(db: WaseelDB, patient_id: str, medication_id: str, medication_name: str,
                        dosage_text: str, start_time: Optional[datetime] = None,
                        num_reminders: int = 3) -> tuple:
    interval_hours = parse_dosing_interval_hours(dosage_text)
    start_time = start_time or datetime.now()
    reminders = [{
        "reminder_id": f"R-{uuid.uuid4().hex[:6].upper()}",
        "patient_id": patient_id,
        "medication_id": medication_id,
        "medication_name": medication_name,
        "due_time": (start_time + timedelta(hours=interval_hours * i)).isoformat(),
        "fired": False,
    } for i in range(1, num_reminders + 1)]
    db.add_reminders(reminders)
    return reminders, interval_hours

## Section 8 — Post-Dose Check-In + AI Daily Summary
**ITU-T Y.3172 stage: SRC (symptom input) → M (grounded LLM summary) → SINK (voice + alert).**

Uses the knowledge base's `detectableBy` tagging: symptoms tagged `"patient"` are
asked directly (things a blind patient can feel — pain, itching, dizziness).
Symptoms tagged `"caregiver_or_touch"` are visual (rash, swelling) — the patient
is never asked to self-assess those; instead a caregiver visual-check flag is set.

In [ ]:
def build_checkin_prompts(matched_entry: dict) -> tuple:
    all_signs = matched_entry.get("sideEffects", []) + matched_entry.get("seriousWarningSigns", [])
    patient_detectable = [s["symptom"] for s in all_signs if s.get("detectableBy") == "patient"]
    caregiver_detectable = [s["symptom"] for s in all_signs if s.get("detectableBy") != "patient"]
    return patient_detectable, caregiver_detectable


def post_dose_checkin(db: WaseelDB, client, model_name: str, patient_id: str,
                       log_entry: dict, matched_entry: dict, use_voice: bool = True):
    """Runs a few hours after a dose is logged: asks the patient (voice-first, in
    Saudi dialect) only about symptoms they could actually feel themselves."""
    patient_detectable, caregiver_detectable = build_checkin_prompts(matched_entry)

    if patient_detectable:
        question_text = "قبل شوي أخذت دواءك، حاسس بأي من هالأعراض؟ " + "، أو ".join(patient_detectable) + "؟"
    else:
        question_text = "قبل شوي أخذت دواءك، كيف حاسس؟ في أي إزعاج أو أعراض غريبة؟"

    print(f"🩺 Check-in question: {question_text}")
    audio = synthesize_speech(question_text, output_path="checkin_question.mp3")
    play_audio(audio)

    if use_voice:
        recording = record_audio("checkin_response.wav", seconds=6)
        symptoms_text = transcribe_arabic_speech(recording)
    else:
        symptoms_text = input("اكتب وصف الأعراض: ")

    try:
        severity_score = int(input("Severity 0-10 (patient's self-report, collected via app slider in production): "))
    except ValueError:
        severity_score = 0

    caregiver_visual_check_needed = bool(caregiver_detectable) and severity_score > 0

    result = db.log_medication(
        patient_id=patient_id,
        medication_id=log_entry["medication_id"],
        medication_name=log_entry["medication_name"],
        scheduled_time=log_entry["scheduled_time"],
        actual_time=datetime.now().isoformat(),
        status="taken",
        symptoms_reported=symptoms_text,
        severity_score=severity_score,
        caregiver_visual_check_needed=caregiver_visual_check_needed,
    )
    db.log_audit_event("post_dose_checkin", patient_id, {
        "log_id": log_entry["log_id"], "symptoms_text": symptoms_text, "severity_score": severity_score,
    }, compliance_rule_ids=["SFDA-REG-003"])

    return result


def generate_daily_summary(db: WaseelDB, client, model_name: str, patient_id: str) -> str:
    """Grounded (not hallucinated) end-of-day summary built only from what's
    actually in today's medication_logs for this patient."""
    today = datetime.now().date().isoformat()
    todays_logs = [l for l in db.get_medication_history(patient_id) if l["scheduled_time"][:10] == today]

    if not todays_logs:
        return "ما فيه جرعات مسجلة اليوم."

    lines = []
    for l in todays_logs:
        status_ar = "أخذها" if l["status"] == "taken" else "فاتته"
        lines.append(f"- {l['medication_name']}: {status_ar}، تأخير {l['delay_minutes']} دقيقة، "
                      f"شدة الأعراض {l['severity_score']}/10، الأعراض: {l['symptoms_reported'] or 'ما ذكر شي'}")
    grounded_facts = "\n".join(lines)

    user_prompt = f"""هذا سجل جرعات المريض اليوم (بيانات حقيقية من قاعدة البيانات، لا تخترع شي غيرها):
{grounded_facts}

اكتب ملخص قصير وودود باللهجة السعودية عن يوم المريض مع أدويته، مبني فقط على البيانات أعلاه."""

    fallback = "ملخص يومك: " + " ".join(lines)
    summary = call_llm(client, model_name, SYSTEM_PROMPT, user_prompt, fallback)
    db.log_audit_event("daily_summary_generated", patient_id, {"summary": summary},
                        compliance_rule_ids=["SFDA-REG-002", "SFDA-REG-003"])
    return summary

## Section 9 — Full Orchestration: the end-to-end patient scenario
Implements, in order: biometric sign-in → caregiver sync → log medication
(voice or camera, with capture guidance) → auto-scheduled reminders →
post-dose check-in → AI daily summary.

In [ ]:
def log_medication_by_camera(db, client, model_name, knowledge_base, patient_id):
    photo_path = capture_with_guidance()
    image = cv2.imread(photo_path)
    processed = preprocess_image(image)
    details = extract_medicine_details(processed)
    os.remove(photo_path)
    print(f"OCR detected: {details['all_detected']} (avg confidence {details['avg_confidence']:.2f})")

    match, score, matched_line = find_best_match_text(details["all_detected"], knowledge_base)
    if details["avg_confidence"] < 0.4:
        match = None
    return _finish_medication_log(db, client, model_name, patient_id, match)


def log_medication_by_voice(db, client, model_name, knowledge_base, patient_id):
    print("🎙️ قل اسم الدواء اللي تبي تسجله...")
    recording = record_audio("med_voice_query.wav", seconds=5)
    query_text = transcribe_arabic_speech(recording)
    print(f"Heard: {query_text!r}")
    match, score, matched_line = find_best_match_text([query_text], knowledge_base)
    return _finish_medication_log(db, client, model_name, patient_id, match)


def _finish_medication_log(db, client, model_name, patient_id, match):
    explanation_text, matched_entry, confidence = get_medication_explanation(match, client, model_name)
    print("📝 LLM explanation (Saudi Arabic):", explanation_text)
    audio_path = synthesize_speech(explanation_text)
    play_audio(audio_path)

    db.log_audit_event("medication_explanation_generated", patient_id, {
        "matched_medication_id": matched_entry["id"] if matched_entry else None,
        "confidence_score": confidence,
    }, compliance_rule_ids=["SFDA-REG-002", "SFDA-REG-003"])

    if not matched_entry:
        print("Nothing logged — no confident medication match.")
        return None, None

    scheduled_time = datetime.now().isoformat()
    result = db.log_medication(
        patient_id=patient_id, medication_id=matched_entry["id"],
        medication_name=get_trade_name(matched_entry, "en"),
        scheduled_time=scheduled_time, actual_time=scheduled_time, status="taken",
    )
    if result["alert_triggered"]:
        print("🚨 HIGH RISK — caregiver SMS alert triggered.")

    reminders, interval_hours = schedule_reminders(
        db, patient_id, matched_entry["id"], get_trade_name(matched_entry, "en"),
        matched_entry.get("dosageInstructions_saudi_dialect", ""), start_time=datetime.now(),
    )
    print(f"⏰ Scheduled {len(reminders)} reminders every {interval_hours}h.")
    return result["log"], matched_entry

## Section 10 — Demo run
Set `ANTHROPIC_API_KEY` for a live LLM, or leave it unset to run in DEMO_MODE
(grounded template responses — the whole flow still works end to end).
Upload `waseel_knowledge_base.json` to the Colab session before running.

In [ ]:
if __name__ == "__main__":
    # --- Compliance rules (Code Block 1) ---
    collector = ITUDataCollector(dataset_name="SFDA_AI_Readiness_v1")
    rules = ITUPreprocessing().process(collector.collect_sfda_rules())
    print(json.dumps(rules, indent=2))

    # --- LLM client (optional; DEMO_MODE if absent) ---
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    client, MODEL_NAME = None, "claude-sonnet-4-6"
    if api_key:
        from anthropic import Anthropic
        client = Anthropic(api_key=api_key)
    else:
        print("No ANTHROPIC_API_KEY set — running in DEMO_MODE (grounded templates instead of live LLM).")

    # --- Knowledge base + DB ---
    knowledge_base = load_knowledge_base("waseel_knowledge_base.json")
    db = WaseelDB("waseel_runtime_db.json")

    # --- Enroll & sign in a demo patient ---
    patient_id = db.add_patient("مريض تجريبي")
    db.set_alert_threshold(patient_id, missed_dose_minutes=30, severity_threshold=6)

    auth = BiometricAuth()
    auth.enroll_face(patient_id)
    signed_in = auth.sign_in(patient_id, method="face")

    if signed_in:
        # --- Caregiver sync ---
        if auth.sign_in(patient_id, method="face"):  # re-verify before sharing access
            code, expires_at = db.create_pairing_code(patient_id)
            print(f"🔗 Caregiver pairing code: {code} (expires {expires_at})")
            redeem_result = db.redeem_pairing_code(code, "مرافق تجريبي", "+9665XXXXXXXX", "ابن/ابنة")
            print(redeem_result)

        # --- Log a medication (camera or voice) ---
        log_entry, matched_entry = log_medication_by_camera(db, client, MODEL_NAME, knowledge_base, patient_id)

        # --- A few hours later: post-dose check-in ---
        if log_entry and matched_entry:
            post_dose_checkin(db, client, MODEL_NAME, patient_id, log_entry, matched_entry, use_voice=False)

        # --- AI daily summary ---
        summary = generate_daily_summary(db, client, MODEL_NAME, patient_id)
        print("\n📋 Daily summary:", summary)
        summary_audio = synthesize_speech(summary, output_path="daily_summary.mp3")
        play_audio(summary_audio)